In [1]:
import torch
from torch.utils.data import random_split, DataLoader, Dataset

class ProteinDDGDataset(Dataset):
    """
    A Dataset class that contains wild-type sequence embeddings,
    mutant-type sequence embeddings and corresponding DDG values.
    
    """
    def __init__(self, wild_embeddings, mutant_embeddings, ddg):
        self.wild = torch.tensor(wild_embeddings, dtype=torch.float32)
        self.mut = torch.tensor(mutant_embeddings, dtype=torch.float32)
        self.ddg = torch.tensor(ddg, dtype=torch.float32)

    def __len__(self):
        return len(self.ddg)
    
    def __getitem__(self, idx):
        return self.wild[idx], self.mut[idx], self.ddg[idx]
    
def create_loader(data, subset, batch_size, shuffle=False):
    """
    Creates and returns a DataLoader of a
    subset from a larger dataset
    """
    split = ProteinDDGDataset(
        data['wild_embeddings'][subset.indices],
        data['mutant_embeddings'][subset.indices],
        data['ddg'][subset.indices]
    )

    return DataLoader(split, batch_size, shuffle)

In [2]:
from tqdm import tqdm
from torch.utils.tensorboard import SummaryWriter
import torch
import torch.nn as nn
import torch.nn.functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'

def train_model(model_class, train_loader, val_loader, model_name, epochs=100):
    model = model_class().to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
    outputs_path = f'../../outputs/regression/{model_name}'
    writer = SummaryWriter(outputs_path)

    for epoch in tqdm(range(epochs)):
        # Training
        model.train()
        train_loss = 0.0
        all_preds_train = []    # Store predictions
        all_targets_train = []  # Store ground truth

        for wild, mut, ddg in train_loader:
            wild, mut, ddg = wild.to(device), mut.to(device), ddg.to(device)
            optimizer.zero_grad()
            outputs = model(wild, mut)
            loss = criterion(outputs, ddg)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * wild.size(0)
            
            # Store batch predictions and targets
            all_preds_train.append(outputs.detach().cpu())
            all_targets_train.append(ddg.detach().cpu())

        # Concatenate all batches
        all_preds_train = torch.cat(all_preds_train).flatten()
        all_targets_train = torch.cat(all_targets_train).flatten()
        
        # Calculate training metrics
        train_loss /= len(train_loader.dataset)
        train_rmse = torch.sqrt(torch.tensor(train_loss))
        train_pearson = torch.corrcoef(torch.stack([all_preds_train, all_targets_train]))[0, 1].item()

        # Validation
        model.eval()
        val_loss = 0.0
        all_preds_val = []  # Store predictions
        all_targets_val = []  # Store ground truth
        
        with torch.no_grad():
            for wild, mut, ddg in val_loader:
                wild, mut, ddg = wild.to(device), mut.to(device), ddg.to(device)
                outputs = model(wild, mut)
                loss = criterion(outputs, ddg)
                val_loss += loss.item() * wild.size(0)
                
                # Store batch predictions and targets
                all_preds_val.append(outputs.detach().cpu())
                all_targets_val.append(ddg.detach().cpu())

        # Concatenate all batches
        all_preds_val = torch.cat(all_preds_val).flatten()
        all_targets_val = torch.cat(all_targets_val).flatten()
        
        # Calculate validation metrics
        val_loss /= len(val_loader.dataset)
        val_rmse = torch.sqrt(F.mse_loss(all_preds_val, all_targets_val)).item()
        val_pearson = torch.corrcoef(torch.stack([all_preds_val, all_targets_val]))[0, 1].item()

        # Log all metrics to TensorBoard
        writer.add_scalars('MSE', {'train': train_loss, 'val': val_loss}, epoch)
        writer.add_scalars('RMSE', {'train': train_rmse, 'val': val_rmse}, epoch)
        writer.add_scalars('Pearson', {'train': train_pearson, 'val': val_pearson}, epoch)
        
        # Log histograms of parameters
        if epoch % 10 == 0:
            for name, param in model.named_parameters():
                writer.add_histogram(name, param, epoch)

        # Print statistics
        print(f'Epoch {epoch+1}/{epochs} | '
              f'Train MSE: {train_loss:.4f} | Val MSE: {val_loss:.4f} | '
              f'Train RMSE: {train_rmse:.4f} | Val RMSE: {val_rmse:.4f} | '
              f'Train Pearson: {train_pearson:.4f} | Val Pearson: {val_pearson:.4f}')
        
        # Save the last five models
        if epoch in list(range(epochs -5, epochs)):
            torch.save(model.state_dict(), outputs_path + f'/checkpoint-epoch-{epoch+1}.pt')

    writer.close()
    del model
    torch.cuda.empty_cache()

In [3]:
from torch.utils.data import random_split
import gc

def training_loop(dataset_path, model_class, model_name, batch_size, epochs=100):
    """The main training loop method"""
    # Load the training data
    data = torch.load(dataset_path)
    
    dataset = ProteinDDGDataset(
        wild_embeddings=data['wild_embeddings'],
        mutant_embeddings=data['mutant_embeddings'],
        ddg=data['ddg']
    )

    # Split dataset in a 9:1 ratio
    train_size = int(0.90 * len(dataset))
    test_size = len(dataset) - train_size

    # Generate reproducible split
    generator = torch.Generator().manual_seed(42)

    # Generate the subsets of the splits
    train_subset, eval_subset = random_split(
        dataset=dataset, 
        lengths=[train_size, test_size],
        generator=generator
    )

    # Create the DataLoaders for the splits
    train_loader = create_loader(data, train_subset, batch_size, shuffle=True)
    eval_loader = create_loader(data, eval_subset, batch_size, shuffle=False)

    # Log dataset and subset metrics
    print(f"Total samples: {len(dataset)}")
    print(f"Train: {len(train_subset)} ({len(train_subset)/len(dataset):.1%})")
    print(f"Validation: {len(eval_subset)} ({len(eval_subset)/len(dataset):.1%})")

    del dataset
    gc.collect()
    torch.cuda.empty_cache()

    # And at last, train the model!
    train_model(model_class, train_loader, eval_loader, model_name, epochs)

In [4]:
from torch import nn

# Training on 320D ESM2-8M embeddings

class DDGPredictor8M(nn.Module):
    def __init__(self):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(320, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

    def forward(self, wild, mut):
        diff = mut - wild
        return self.head(diff).squeeze()
    
training_loop(
    dataset_path='../../data/training/regression/spysmith_8M.pt',
    model_class=DDGPredictor8M,
    model_name='SpySmith-8M',
    batch_size=64,
    epochs=100
)

C:\Users\dante\AppData\Local\Temp\ipykernel_30088\3432288791.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.wild = torch.tensor(wild_embeddings, dtype=torch.float32)
C:\Users\dante\AppData\Local\Temp\ipykernel_30088\3432288791.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.mut = torch.tensor(mutant_embeddings, dtype=torch.float32)
C:\Users\dante\AppData\Local\Temp\ipykernel_30088\3432288791.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.ddg = torch.tensor(ddg, dtype=torch.float32)


Total samples: 650936
Train: 585842 (90.0%)
Validation: 65094 (10.0%)


  1%|          | 1/100 [00:39<1:04:47, 39.27s/it]

Epoch 1/100 | Train MSE: 1.1326 | Val MSE: 0.9524 | Train RMSE: 1.0642 | Val RMSE: 0.9759 | Train Pearson: 0.5912 | Val Pearson: 0.6751


  2%|▏         | 2/100 [01:18<1:04:00, 39.19s/it]

Epoch 2/100 | Train MSE: 0.9100 | Val MSE: 0.8238 | Train RMSE: 0.9539 | Val RMSE: 0.9076 | Train Pearson: 0.6909 | Val Pearson: 0.7265


... (192 outputs left out)

 99%|█████████▉| 99/100 [1:07:37<00:41, 41.01s/it]

Epoch 99/100 | Train MSE: 0.3748 | Val MSE: 0.4043 | Train RMSE: 0.6122 | Val RMSE: 0.6359 | Train Pearson: 0.8858 | Val Pearson: 0.8773


100%|██████████| 100/100 [1:08:18<00:00, 40.99s/it]

Epoch 100/100 | Train MSE: 0.3746 | Val MSE: 0.4046 | Train RMSE: 0.6120 | Val RMSE: 0.6361 | Train Pearson: 0.8859 | Val Pearson: 0.8773


In [ ]:
from torch import nn

# Training on 480D ESM2-35M embeddings

class DDGPredictor35M(nn.Module):
    def __init__(self):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(480, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

    def forward(self, wild, mut):
        diff = mut - wild
        return self.head(diff).squeeze()
    
training_loop(
    dataset_path='../../data/training/regression/spysmith_35M.pt',
    model_class=DDGPredictor35M,
    model_name='SpySmith-35M',
    batch_size=64,
    epochs=100
)

C:\Users\dante\AppData\Local\Temp\ipykernel_11208\3432288791.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.wild = torch.tensor(wild_embeddings, dtype=torch.float32)
C:\Users\dante\AppData\Local\Temp\ipykernel_11208\3432288791.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.mut = torch.tensor(mutant_embeddings, dtype=torch.float32)
C:\Users\dante\AppData\Local\Temp\ipykernel_11208\3432288791.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.ddg = torch.tensor(ddg, dtype=torch.float32)


Total samples: 650914
Train: 585822 (90.0%)
Validation: 65092 (10.0%)


  1%|          | 1/100 [01:14<2:02:37, 74.32s/it]

Epoch 1/100 | Train MSE: 0.9642 | Val MSE: 0.8124 | Train RMSE: 0.9819 | Val RMSE: 0.9014 | Train Pearson: 0.6674 | Val Pearson: 0.7325


  2%|▏         | 2/100 [02:30<2:03:04, 75.35s/it]

Epoch 2/100 | Train MSE: 0.7705 | Val MSE: 0.7077 | Train RMSE: 0.8778 | Val RMSE: 0.8412 | Train Pearson: 0.7462 | Val Pearson: 0.7755


... (192 outputs left out)

 99%|█████████▉| 99/100 [2:03:15<01:14, 74.84s/it]

Epoch 99/100 | Train MSE: 0.2975 | Val MSE: 0.3422 | Train RMSE: 0.5455 | Val RMSE: 0.5850 | Train Pearson: 0.9104 | Val Pearson: 0.8977


100%|██████████| 100/100 [2:04:31<00:00, 74.72s/it]

Epoch 100/100 | Train MSE: 0.2964 | Val MSE: 0.3462 | Train RMSE: 0.5444 | Val RMSE: 0.5884 | Train Pearson: 0.9108 | Val Pearson: 0.8960


In [5]:
from torch import nn

# Training on 640D ESM2-150M embeddings

class DDGPredictor150M(nn.Module):
    def __init__(self):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(640, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

    def forward(self, wild, mut):
        diff = mut - wild
        return self.head(diff).squeeze()
    
training_loop(
    dataset_path='../../data/training/regression/spysmith_150M.pt',
    model_class=DDGPredictor150M,
    model_name='SpySmith-150M',
    batch_size=16,
    epochs=100
)

C:\Users\dante\AppData\Local\Temp\ipykernel_11208\3432288791.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.wild = torch.tensor(wild_embeddings, dtype=torch.float32)
C:\Users\dante\AppData\Local\Temp\ipykernel_11208\3432288791.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.mut = torch.tensor(mutant_embeddings, dtype=torch.float32)
C:\Users\dante\AppData\Local\Temp\ipykernel_11208\3432288791.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.ddg = torch.tensor(ddg, dtype=torch.float32)


Total samples: 650914
Train: 585822 (90.0%)
Validation: 65092 (10.0%)


  1%|          | 1/100 [02:21<3:53:42, 141.64s/it]

Epoch 1/100 | Train MSE: 0.8508 | Val MSE: 0.6935 | Train RMSE: 0.9224 | Val RMSE: 0.8327 | Train Pearson: 0.7146 | Val Pearson: 0.7782


  2%|▏         | 2/100 [04:49<3:57:21, 145.32s/it]

Epoch 2/100 | Train MSE: 0.6529 | Val MSE: 0.6121 | Train RMSE: 0.8080 | Val RMSE: 0.7823 | Train Pearson: 0.7902 | Val Pearson: 0.8100


... (192 outputs left out)

 99%|█████████▉| 99/100 [4:00:04<02:23, 143.10s/it]

Epoch 99/100 | Train MSE: 0.2705 | Val MSE: 0.3198 | Train RMSE: 0.5201 | Val RMSE: 0.5655 | Train Pearson: 0.9189 | Val Pearson: 0.9046


100%|██████████| 100/100 [4:02:31<00:00, 145.51s/it]

Epoch 100/100 | Train MSE: 0.2696 | Val MSE: 0.3153 | Train RMSE: 0.5192 | Val RMSE: 0.5615 | Train Pearson: 0.9192 | Val Pearson: 0.9056


In [ ]:
from torch import nn

# Training on 1280D ESM2-650M embeddings

class DDGPredictor650M(nn.Module):
    def __init__(self):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(1280, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, wild, mut):
        diff = mut - wild
        return self.head(diff).squeeze()
    
training_loop(
    dataset_path='../../data/training/regression/spysmith_650M.pt',
    model_class=DDGPredictor650M,
    model_name='SpySmith-650M',
    batch_size=2,
    epochs=100
)

C:\Users\dante\AppData\Local\Temp\ipykernel_24744\3432288791.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.wild = torch.tensor(wild_embeddings, dtype=torch.float32)
C:\Users\dante\AppData\Local\Temp\ipykernel_24744\3432288791.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.mut = torch.tensor(mutant_embeddings, dtype=torch.float32)
C:\Users\dante\AppData\Local\Temp\ipykernel_24744\3432288791.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.ddg = torch.tensor(ddg, dtype=torch.float32)


Total samples: 650914
Train: 585822 (90.0%)
Validation: 65092 (10.0%)


  1%|          | 1/100 [14:53<24:33:30, 893.03s/it]

Epoch 1/100 | Train MSE: 0.7291 | Val MSE: 0.5967 | Train RMSE: 0.8539 | Val RMSE: 0.7725 | Train Pearson: 0.7620 | Val Pearson: 0.8142


  2%|▏         | 2/100 [30:08<24:39:55, 906.08s/it]

Epoch 2/100 | Train MSE: 0.5523 | Val MSE: 0.5135 | Train RMSE: 0.7432 | Val RMSE: 0.7166 | Train Pearson: 0.8260 | Val Pearson: 0.8417


  3%|▎         | 3/100 [45:21<24:29:45, 909.13s/it]

Epoch 3/100 | Train MSE: 0.4968 | Val MSE: 0.4779 | Train RMSE: 0.7048 | Val RMSE: 0.6913 | Train Pearson: 0.8451 | Val Pearson: 0.8538
